# E8 — Liquidity Migration W2/W3/W4

In [1]:
import sys, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
sys.path.insert(0, '.')
sys.path.insert(0, '../notebooks')
from config import (RESULTS_DIR, SEAGATE_DIR, FIGS_DIR, WINDOWS_META,
                    BASELINE_STATS, UPDATED_STATS, WIN_COLORS,
                    TICK, MULT, save_fig)
Path('figures').mkdir(exist_ok=True)

# SEAGATE required for E notebooks


In [2]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, wk in zip(axes, ['W2','W3','W4']):
    wm = WINDOWS_META[wk]
    front_sz, back_sz = [], []
    for day in wm['days']:
        row_f, row_b = np.nan, np.nan
        for sym, store in [(wm['front'], front_sz), (wm['back'], back_sz)]:
            fpath = list(SEAGATE_DIR.glob(f'mbp10_*{sym}*{day}*.parquet'))
            if not fpath: store.append(np.nan); continue
            df = pd.read_parquet(fpath[0], columns=['ts_event','bid_sz_00','ask_sz_00'])
            df['ts_event'] = pd.to_datetime(df['ts_event'])
            rth = ((df['ts_event'].dt.hour*60+df['ts_event'].dt.minute) >= wm['rth_start_min']) &                   ((df['ts_event'].dt.hour*60+df['ts_event'].dt.minute) <= wm['rth_end_min'])
            df = df[rth]
            sz = ((df['bid_sz_00']+df['ask_sz_00'])/2).median()
            store.append(sz)
            del df; gc.collect()
    x = range(len(wm['days']))
    ax.plot(x, front_sz, marker='o', label=wm['front'], color='steelblue')
    ax.plot(x, back_sz,  marker='s', label=wm['back'],  color='darkorange')
    ax.set_xticks(list(x)); ax.set_xticklabels(wm['day_labels'], fontsize=8)
    ax.set_title(wk)
    ax.set_ylabel('Median Top-of-Book Size (contracts)')
    ax.legend()
fig.suptitle('Liquidity Migration: Median Bid/Ask Size — W2/W3/W4', fontsize=13)
save_fig(fig, 'E8_liq_migration_w2w3w4.png')


  Saved --> figures/E8_liq_migration_w2w3w4.png
